# 판매금액순 top50 상품정보

In [1]:
# 라이브러리
import requests
import pandas as pd
import time
import json

In [ ]:
# 테스트 코드
BASE_URL = "https://api.musinsa.com/api2/dp/v2/plp/goods"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/brand/musinsastandardwoman/products?gf=A&sortCode=SALE_ONE_MONTH_AMOUNT",
    "Accept": "application/json",
}

all_products = []

for page in range(1, 3):
    params = {
        "gf": "A",
        "sortCode": "SALE_ONE_MONTH_AMOUNT",
        "brand": "musinsastandardwoman",
        "size": 30,
        "caller": "FLAGSHIP",
        "page": page
    }

    res = requests.get(BASE_URL, params=params, headers=HEADERS)
    print(f"Page {page} 상태코드: {res.status_code}")

    # 403이면 건너뜀
    if res.status_code != 200:
        print(f"Page {page} 접근 불가, 스킵")
        continue

    data = res.json()

    # JSON 구조 확인 (처음 한 번만)
    if page == 1:
        print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])

    # 구조에 따라 list 위치 찾기
    goods_list = data.get("data", {}).get("list", [])
    print(f"Page {page} 상품 수: {len(goods_list)}")

    for rank, item in enumerate(goods_list, start=(page-1)*30 + 1):
        all_products.append({
            "rank":         rank,
            "goods_no":     item.get("goodsNo"),
            "goods_name":   item.get("goodsName"),
            "category":     item.get("category"),
            "price":        item.get("normalPrice"),
            "sale_price":   item.get("salePrice"),
            "discount_rate":item.get("discountRate"),
            "review_count": item.get("reviewCount"),
            "rating":       item.get("rating"),
            "product_url":  f"https://www.musinsa.com/products/{item.get('goodsNo')}",
        })

    time.sleep(2)

print(f"\n총 수집: {len(all_products)}개")

In [ ]:
# 수집 코드
BASE_URL = "https://api.musinsa.com/api2/dp/v2/plp/goods"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/brand/musinsastandardwoman/products?gf=A&sortCode=SALE_ONE_MONTH_AMOUNT",
    "Accept": "application/json",
}

params = {
    "gf": "A",
    "sortCode": "SALE_ONE_MONTH_AMOUNT",
    "brand": "musinsastandardwoman",
    "size": 50,   # ← 50개 한번에
    "caller": "FLAGSHIP",
    "page": 1
}

res = requests.get(BASE_URL, params=params, headers=HEADERS)
print(f"상태코드: {res.status_code}")

data = res.json()
goods_list = data.get("data", {}).get("list", [])
print(f"수집된 상품 수: {len(goods_list)}")

all_products = []
for rank, item in enumerate(goods_list, start=1):
    all_products.append({
        "rank":          rank,
        "goods_no":      item.get("goodsNo"),
        "goods_name":    item.get("goodsName"),
        "price":         item.get("normalPrice"),
        "sale_price":    item.get("price"),
        "sale_rate":     item.get("saleRate"),
        "review_count":  item.get("reviewCount"),
        "review_score":  item.get("reviewScore"),  # 100점 만점
        "product_url":   item.get("goodsLinkUrl"),
    })

df = pd.DataFrame(all_products[:50])
print(df)

df.to_csv("musinsastandard_woman_top50.csv", index=False, encoding="utf-8-sig")
print("저장 완료!")

### 실행 결과 (2026-04-20 수집)

**판매금액순 TOP50 상품정보 수집**

| 항목 | 결과 |
|---|---|
| 상태코드 | 200 |
| 수집된 상품 수 | **50개** |
| 저장 파일 | `musinsastandard_woman_top50.csv` |

**API 응답 구조 확인 (테스트 코드 실행 결과)**
- Page 1 상태코드: 200 → 상품 30개 수집 확인
- Page 2 상태코드: 403 → 접근 불가 (size=50 단일 요청 방식으로 전환)

**수집 데이터 샘플 (상위 5개)**

| rank | goods_no | goods_name | price | sale_price | sale_rate | review_count | review_score |
|---|---|---|---|---|---|---|---|
| 1 | 6092190 | [한소희 PICK] [UV 프로텍션] 우먼즈 시어 윈드브레이커 (5 colors) | 55,900 | 55,900 | 0% | 140 | 98 |
| 2 | 6092187 | [한소희 PICK] 우먼즈 코튼 커브드 팬츠 (3 colors) | 49,900 | 47,410 | 5% | 381 | 96 |
| 3 | 1618312 | 우먼즈 오버사이즈 블레이저 [블랙] | 89,900 | 44,950 | 50% | 12,807 | 98 |
| 4 | 3753637 | [한소희 PICK] 우먼즈 슬림 크루 넥 티셔츠 [화이트] | 15,900 | 14,790 | 7% | 4,719 | 96 |
| 5 | 1803462 | [한소희 PICK] 우먼즈 베이식 블레이저 [블랙] | 89,900 | 62,930 | 30% | 4,098 | 98 |

> **테스트 → 최종 전환 이유:**
> 30개씩 페이지네이션 방식은 Page 2에서 403 오류 발생
> → size=50으로 단일 요청하는 방식으로 변경하여 50개 전체 수집 완료

# 판매금액순 top50 리뷰

In [ ]:
# TOP 50 CSV에서 상품번호 불러오기
df_products = pd.read_csv("musinsastandard_woman_top50.csv")
goods_no_list = df_products["goods_no"].tolist()
print(f"수집 대상 상품 수: {len(goods_no_list)}개")

In [ ]:
# 키 구조 확인
import json

REVIEW_URL = "https://goods.musinsa.com/api2/review/v1/view/list"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/",
}

params = {
    "page": 0,
    "pageSize": 20,
    "goodsNo": 1417691,
    "sort": "new_desc",
    "selectedSimilarNo": 1417691,
    "myFilter": "false",
    "hasPhoto": "false",
    "isExperience": "false"
}

res = requests.get(REVIEW_URL, params=params, headers=HEADERS)
print(f"상태코드: {res.status_code}")

data = res.json()

# 전체 구조 키 확인
print("최상위 키:", data.keys())
print(json.dumps(data, indent=2, ensure_ascii=False)[:1000])

In [ ]:
# 수집 코드
REVIEW_URL = "https://goods.musinsa.com/api2/review/v1/view/list"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/",
}

all_reviews = []

for idx, goods_no in enumerate(goods_no_list):
    print(f"\n[{idx+1}/50] 상품 {goods_no} 리뷰 수집 중...")
    page = 0
    collected = 0

    while collected < 1000:
        params = {
            "page": page,
            "pageSize": 20,
            "goodsNo": goods_no,
            "sort": "new_desc",
            "selectedSimilarNo": goods_no,
            "myFilter": "false",
            "hasPhoto": "false",
            "isExperience": "false"
        }

        res = requests.get(REVIEW_URL, params=params, headers=HEADERS)

        if res.status_code != 200:
            print(f"  오류: {res.status_code}, 스킵")
            break

        data = res.json()
        reviews = data.get("data", {}).get("list", [])  # ← 여기 수정!

        if not reviews:
            break

        for r in reviews:
            size_answer = ""
            survey = r.get("reviewSurveySatisfaction") or {}
            for q in survey.get("questions", []):
                if q.get("attribute") == "사이즈":
                    answers = q.get("answers", [])
                    if answers:
                        size_answer = answers[0].get("answerShortText", "")

            profile = r.get("userProfileInfo") or {}

            all_reviews.append({
                "goods_no":      goods_no,
                "review_no":     r.get("no"),
                "content":       r.get("content"),
                "grade":         r.get("grade"),
                "option_size":   r.get("goodsOption"),
                "size_feedback": size_answer,
                "create_date":   r.get("createDate"),
                "review_sex":    profile.get("reviewSex"),
                "height":        profile.get("userHeight"),
                "weight":        profile.get("userWeight"),
            })

        collected += len(reviews)
        total = data.get("data", {}).get("total", 0)  # ← 여기도 수정!

        if collected >= total or collected >= 1000:
            break

        page += 1
        time.sleep(1.5)

    print(f"  완료: {collected}건")

# 저장
df_reviews = pd.DataFrame(all_reviews)
df_reviews.to_csv("musinsastandard_reviews_raw.csv", index=False, encoding="utf-8-sig")
print(f"\n전체 수집 완료: {len(df_reviews)}건")
print(df_reviews.head())

### 실행 결과 (2026-04-20 수집)

**판매금액순 TOP50 리뷰 수집**

| 항목 | 결과 |
|---|---|
| 수집 대상 | 50개 상품 |
| 전체 수집 리뷰 | **28,216건** |
| 수집 기준 | 최신순, 상품당 최대 1,000건 |
| 저장 파일 | `musinsastandard_reviews_raw.csv` |

**API 구조 확인 (키 구조 탐색 결과)**
- 최상위 키: `data`, `meta`
- 리뷰 목록 위치: `data > list`
- 사이즈 피드백 위치: `reviewSurveySatisfaction > questions > answers`
- 작성자 체형 위치: `userProfileInfo > userHeight / userWeight`

**상품별 수집 결과**

| 순번 | goods_no | 수집 건수 |
|---|---|---|
| 1 | 6092190 | 140건 |
| 2 | 6092187 | 381건 |
| 3 | 1618312 | 1,000건 |
| 4 | 3753637 | 1,000건 |
| 5 | 1803462 | 1,000건 |
| 6 | 2724650 | 1,000건 |
| 7 | 3758435 | 691건 |
| 8 | 5829495 | 56건 |
| 9 | 2820939 | 1,000건 |
| 10 | 5721301 | 44건 |
| 11 | 5671241 | 48건 |
| 12 | 2818150 | 1,000건 |
| 13 | 5287054 | 316건 |
| 14 | 4651725 | 289건 |
| 15 | 1243054 | 1,000건 |
| 16 | 2149536 | 1,000건 |
| 17 | 1417691 | 1,000건 |
| 18 | 4651407 | 377건 |
| 19 | 1596414 | 1,000건 |
| 20 | 2978126 | 1,000건 |
| 21 | 2978123 | 1,000건 |
| 22 | 5721302 | 25건 |
| 23 | 3425498 | 397건 |
| 24 | 5166592 | 172건 |
| 25 | 5151475 | 156건 |
| 26 | 3052664 | 1,000건 |
| 27 | 1357769 | 1,000건 |
| 28 | 5721297 | 30건 |
| 29 | 3445169 | 1,000건 |
| 30 | 3790852 | 875건 |
| 31 | 2820940 | 1,000건 |
| 32 | 5661619 | 28건 |
| 33 | 2551390 | 1,000건 |
| 34 | 3323598 | 1,000건 |
| 35 | 3051707 | 472건 |
| 36 | 5671238 | 23건 |
| 37 | 5820431 | 48건 |
| 38 | 1932037 | 1,000건 |
| 39 | 5671244 | 13건 |
| 40 | 5892071 | 24건 |
| 41 | 3753636 | 1,000건 |
| 42 | 3753631 | 1,000건 |
| 43 | 5287052 | 203건 |
| 44 | 4644447 | 159건 |
| 45 | 3417709 | 1,000건 |
| 46 | 5829493 | 38건 |
| 47 | 5661620 | 27건 |
| 48 | 6121803 | 37건 |
| 49 | 2792571 | 1,000건 |
| 50 | 4677944 | 147건 |

> 1,000건 상품: 26개 (리뷰 수가 1,000건 이상인 상품은 최신 1,000건만 수집)
> 1,000건 미만 상품: 24개 (전체 리뷰 수집 완료)

**수집 데이터 샘플 (상위 5개)**

| goods_no | review_no | grade | option_size | size_feedback | create_date | review_sex | height | weight |
|---|---|---|---|---|---|---|---|---|
| 6092190 | 83820239 | 5 | 04.화이트 · S | 정사이즈 | 2026-04-20 | 여성 | 152 | 45 |
| 6092190 | 83817463 | 5 | 02.클라우디 블루 · M | 조금 큼 | 2026-04-20 | 여성 | 162 | 65 |
| 6092190 | 83817336 | 5 | 02.클라우디 블루 · L | 조금 큼 | 2026-04-20 | 여성 | 165 | 62 |
| 6092190 | 83817326 | 5 | 02.클라우디 블루 · L | 조금 큼 | 2026-04-20 | 여성 | 165 | 62 |
| 6092190 | 83817305 | 5 | 03.라이트 그레이 · L | 조금 큼 | 2026-04-20 | 여성 | 165 | 62 |

# 무신사 추천순 top50 상품정보

In [ ]:
# 수집 코드
BASE_URL = "https://api.musinsa.com/api2/dp/v2/plp/goods"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/brand/musinsastandardwoman/products?gf=A&sortCode=POPULAR",
}

params = {
    "gf": "A",
    "sortCode": "POPULAR",
    "brand": "musinsastandardwoman",
    "size": 50,
    "caller": "FLAGSHIP",
    "page": 1
}

res = requests.get(BASE_URL, params=params, headers=HEADERS)
print(f"상태코드: {res.status_code}")

data = res.json()
goods_list = data.get("data", {}).get("list", [])
print(f"수집된 상품 수: {len(goods_list)}")

all_products = []
for rank, item in enumerate(goods_list, start=1):
    all_products.append({
        "rank":         rank,
        "goods_no":     item.get("goodsNo"),
        "goods_name":   item.get("goodsName"),
        "price":        item.get("normalPrice"),
        "sale_price":   item.get("price"),
        "sale_rate":    item.get("saleRate"),
        "review_count": item.get("reviewCount"),
        "review_score": item.get("reviewScore"),
        "product_url":  item.get("goodsLinkUrl"),
    })

    df_recommend = pd.DataFrame(all_products)
df_recommend.to_csv("musinsastandard_woman_recommend_top50.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {len(df_recommend)}개")

### 실행 결과 (2026-04-21 수집)

**무신사 추천순 TOP50 상품정보 수집**

| 항목 | 결과 |
|---|---|
| 상태코드 | 200 |
| 수집된 상품 수 | **50개** |
| 저장 파일 | `musinsastandard_woman_recommend_top50.csv` |

**수집 데이터 샘플 (상위 5개)**

| rank | goods_no | goods_name | price | sale_price | sale_rate | review_count | review_score |
|---|---|---|---|---|---|---|---|
| 1 | 6092187 | [한소희 PICK] 우먼즈 코튼 커브드 팬츠 (3 colors) | 49,900 | 47,410 | 5% | 386 | 96 |
| 2 | 6121801 | 우먼즈 시어 슬림 티셔츠 (5 colors) | 15,900 | 15,100 | 5% | 11 | 86 |
| 3 | 6092188 | [쿨탠다드] 우먼즈 커브드 스웨트 팬츠 (2 colors) | 49,900 | 49,900 | 0% | 5 | 88 |
| 4 | 6092190 | [한소희 PICK] [UV 프로텍션] 우먼즈 시어 윈드브레이커 (5 colors) | 55,900 | 55,900 | 0% | 143 | 98 |
| 5 | 5928482 | [한소희 PICK] 우먼즈 신세틱 레더 도트 스터드 벨트_25mm | 29,900 | 29,900 | 0% | 8 | 98 |

# 무신사 추천순 top50 리뷰

In [ ]:
REVIEW_URL = "https://goods.musinsa.com/api2/review/v1/view/list"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/",
}

# 추천순 50개 CSV에서 직접 불러오기
df_rec = pd.read_csv("musinsastandard_woman_recommend_top50.csv")
goods_no_list_rec = df_rec['goods_no'].tolist()
print(f"수집 대상: {len(goods_no_list_rec)}개")

In [ ]:
REVIEW_URL = "https://goods.musinsa.com/api2/review/v1/view/list"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/",
}

# 추천순 50개 전체
goods_no_list_rec = df_rec['goods_no'].tolist()
print(f"수집 대상: {len(goods_no_list_rec)}개")

all_reviews_rec = []

for idx, goods_no in enumerate(goods_no_list_rec):
    print(f"\n[{idx+1}/50] 상품 {goods_no} 리뷰 수집 중...")
    page = 0
    collected = 0

    while collected < 1000:
        params = {
            "page": page,
            "pageSize": 20,
            "goodsNo": goods_no,
            "sort": "new_desc",
            "selectedSimilarNo": goods_no,
            "myFilter": "false",
            "hasPhoto": "false",
            "isExperience": "false"
        }

        res = requests.get(REVIEW_URL, params=params, headers=HEADERS)

        if res.status_code != 200:
            print(f"  오류: {res.status_code}, 스킵")
            break

        data = res.json()
        reviews = data.get("data", {}).get("list", [])

        if not reviews:
            break

        for r in reviews:
            size_answer = ""
            survey = r.get("reviewSurveySatisfaction") or {}
            for q in survey.get("questions", []):
                if q.get("attribute") == "사이즈":
                    answers = q.get("answers", [])
                    if answers:
                        size_answer = answers[0].get("answerShortText", "")

            profile = r.get("userProfileInfo") or {}

            all_reviews_rec.append({
                "goods_no":      goods_no,
                "review_no":     r.get("no"),
                "content":       r.get("content"),
                "grade":         r.get("grade"),
                "option_size":   r.get("goodsOption"),
                "size_feedback": size_answer,
                "create_date":   r.get("createDate"),
                "review_sex":    profile.get("reviewSex"),
                "height":        profile.get("userHeight"),
                "weight":        profile.get("userWeight"),
            })

        collected += len(reviews)
        total = data.get("data", {}).get("total", 0)

        if collected >= total or collected >= 1000:
            break

        page += 1
        time.sleep(1.5)

    print(f"  완료: {collected}건")

# 저장
df_reviews_rec = pd.DataFrame(all_reviews_rec)
df_reviews_rec.to_csv("musinsastandard_recommend_reviews_raw.csv", index=False, encoding="utf-8-sig")
print(f"\n전체 수집 완료: {len(df_reviews_rec)}건")

### 실행 결과 (2026-04-21 수집)

**무신사 추천순 TOP50 리뷰 수집**

| 항목 | 결과 |
|---|---|
| 수집 대상 | 50개 상품 |
| 전체 수집 리뷰 | **21,295건** |
| 수집 기준 | 최신순, 상품당 최대 1,000건 |

**상품별 수집 결과**

| 순번 | goods_no | 수집 건수 |
|---|---|---|
| 1 | 6092187 | 389건 |
| 2 | 6121801 | 12건 |
| 3 | 6092188 | 5건 |
| 4 | 6092190 | 146건 |
| 5 | 5928482 | 8건 |
| 6 | 6121803 | 40건 |
| 7 | 5985469 | 25건 |
| 8 | 5166562 | 118건 |
| 9 | 4728043 | 223건 |
| 10 | 2477832 | 1,000건 |
| 11 | 2818151 | 1,000건 |
| 12 | 5940961 | 28건 |
| 13 | 5940965 | 19건 |
| 14 | 6092691 | 60건 |
| 15 | 5768119 | 15건 |
| 16 | 3758437 | 598건 |
| 17 | 2978126 | 1,000건 |
| 18 | 2551390 | 1,000건 |
| 19 | 5745829 | 51건 |
| 20 | 4644447 | 159건 |
| 21 | 5768101 | 24건 |
| 22 | 3378573 | 974건 |
| 23 | 3779651 | 567건 |
| 24 | 5166592 | 174건 |
| 25 | 5795928 | 19건 |
| 26 | 3445169 | 1,000건 |
| 27 | 5661619 | 28건 |
| 28 | 1945314 | 1,000건 |
| 29 | 1357768 | 1,000건 |
| 30 | 2792571 | 1,000건 |
| 31 | 2124425 | 1,000건 |
| 32 | 3863280 | 1,000건 |
| 33 | 5258615 | 65건 |
| 34 | 5892071 | 25건 |
| 35 | 5242005 | 61건 |
| 36 | 5657916 | 30건 |
| 37 | 4719640 | 147건 |
| 38 | 5698182 | 7건 |
| 39 | 3753637 | 1,000건 |
| 40 | 5721297 | 31건 |
| 41 | 2820940 | 1,000건 |
| 42 | 2818150 | 1,000건 |
| 43 | 3135346 | 1,000건 |
| 44 | 4624151 | 52건 |
| 45 | 5166563 | 43건 |
| 46 | 3134739 | 639건 |
| 47 | 3779650 | 345건 |
| 48 | 4642930 | 259건 |
| 49 | 1357771 | 1,000건 |
| 50 | 3134752 | 909건 |

> 1,000건 상품: 19개 (리뷰 수가 1,000건 이상인 상품은 최신 1,000건만 수집)
> 1,000건 미만 상품: 31개 (전체 리뷰 수집 완료)